# Step 1 — 75/15/15 split + blend + RF
convnextv2_tiny | tf_efficientnetv2_s | deit3_small + mpnet embeddings

In [ ]:
!pip install timm==1.0.3 albumentations sentence-transformers -q
print('OK')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 86.2 MB/s eta 0:00:00
OK


In [ ]:
import os, gc, json, random, time, warnings, sys
from pathlib import Path
from collections import deque
import cv2
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
warnings.filterwarnings('ignore')
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

PyTorch: 2.11.0+cu128 | GPU: NVIDIA A100-SXM4-40GB


In [ ]:
IS_KAGGLE = os.path.exists('/kaggle/input')
IS_COLAB  = 'google.colab' in sys.modules

DEBUG        = False
DEBUG_EPOCHS = 1

if IS_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/food_step1_tiny') if not DEBUG else Path('/content/food_step1_tiny_debug')
    KAGGLE_DATA = Path('/content/data')
    KAGGLE_DATA.mkdir(parents=True, exist_ok=True)
    if not (KAGGLE_DATA / 'train_labels.csv').exists():
        uploaded = files.upload()
        !mkdir -p ~/.kaggle
        !mv kaggle.json ~/.kaggle/
        !chmod 600 ~/.kaggle/kaggle.json
        !pip install kaggle -q
        !kaggle competitions download -c m2-food-calorie-estimation -p /content/data
        !unzip -q /content/data/m2-food-calorie-estimation.zip -d /content/data/
    DESC_CSV  = Path('/content/drive/MyDrive/food_descriptions/descriptions_merged.csv')
    EMB_CACHE = Path('/content/drive/MyDrive/food_descriptions/embeddings_mpnet.npz')
elif IS_KAGGLE:
    WORK = Path('/kaggle/working')
    for candidate in [
        Path('/kaggle/input/m2-food-calorie-estimation'),
        Path('/kaggle/input/competitions/m2-food-calorie-estimation'),
    ]:
        if (candidate / 'train_labels.csv').exists():
            KAGGLE_DATA = candidate; break
    else:
        KAGGLE_DATA = list(Path('/kaggle/input').rglob('train_labels.csv'))[0].parent
    DESC_CSV  = Path('/kaggle/input/datasets/yacineabdelouhab/food-descriptions-moondream/descriptions_merged.csv')
    EMB_CACHE = Path('/kaggle/working/embeddings_mpnet.npz')
else:
    WORK      = Path('.')
    KAGGLE_DATA = Path(r'C:\Users\Abdel\.cache\kagglehub\competitions\m2-food-calorie-estimation')
    DESC_CSV  = Path(r'C:\Users\Abdel\Desktop\M2Dauphine\DL_Image\Projet\description_textuel\descriptions_merged.csv')
    EMB_CACHE = WORK / 'embeddings_mpnet.npz'

WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
CKPT_DIR = WORK / 'checkpoints'; CKPT_DIR.mkdir(exist_ok=True)
OUT_DIR  = WORK / 'outputs';     OUT_DIR.mkdir(exist_ok=True)

SEED=42; VAL_RATIO=0.15; TEST_RATIO=0.15
PRED_MIN=30.0; PRED_MAX=3600.0
NUM_WORKERS = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, '|', 'debug' if DEBUG else 'normal', '|', WORK)
print('EMB_CACHE:', EMB_CACHE)

Mounted at /content/drive


Saving kaggle.json to kaggle.json
100% 2.06G/2.06G [01:39<00:00, 22.2MB/s]

cuda | normal | /content/drive/MyDrive/food_step1_tiny
EMB_CACHE: /content/drive/MyDrive/food_descriptions/embeddings_mpnet.npz


In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

TRAIN_IMGS = KAGGLE_DATA / 'train' / 'images'
TEST_IMGS  = KAGGLE_DATA / 'test'  / 'images'

def resolve_image_path(img_dir, filename):
    p = img_dir / filename
    if p.exists(): return str(p)
    alt = p.with_suffix('.png' if p.suffix.lower()=='.jpg' else '.jpg')
    return str(alt) if alt.exists() else str(p)

train_df = pd.read_csv(KAGGLE_DATA / 'train_labels.csv')
test_df  = pd.read_csv(KAGGLE_DATA / 'test_ids.csv')
train_df['path'] = train_df['filename'].apply(lambda f: resolve_image_path(TRAIN_IMGS, f))
test_df['path']  = test_df['filename'].apply(lambda f: resolve_image_path(TEST_IMGS, f))
train_df['target_log'] = np.log1p(train_df['calories'].astype(float))
train_df['cal_bin'] = pd.qcut(train_df['calories'], q=10, labels=False, duplicates='drop')

df_temp, df_test_final = train_test_split(train_df, test_size=TEST_RATIO, random_state=SEED, stratify=train_df['cal_bin'])
df_train, df_val = train_test_split(df_temp, test_size=VAL_RATIO/(1-TEST_RATIO), random_state=SEED, stratify=df_temp['cal_bin'])
df_train      = df_train.reset_index(drop=True)
df_val        = df_val.reset_index(drop=True)
df_test_final = df_test_final.reset_index(drop=True)
print(f'train={len(df_train)} val={len(df_val)} test={len(df_test_final)} kaggle={len(test_df)}')

train=2168 val=465 test=465 kaggle=547


In [ ]:
df_desc = pd.read_csv(DESC_CSV)
descriptions = dict(zip(df_desc['image_id'].astype(str), df_desc['description'].fillna('food dish')))
print(f'{len(descriptions)} descriptions')

3645 descriptions


In [ ]:
from sentence_transformers import SentenceTransformer

if EMB_CACHE.exists():
    tmp = np.load(EMB_CACHE, allow_pickle=True)
    if tmp['vecs'].shape[1] != 768:
        EMB_CACHE.unlink()

if EMB_CACHE.exists():
    saved  = np.load(EMB_CACHE, allow_pickle=True)
    id2emb = {k: saved['vecs'][i] for i, k in enumerate(saved['ids'])}
else:
    st_model = SentenceTransformer('all-mpnet-base-v2', device='cpu')
    ids_list = list(descriptions.keys())
    vecs     = st_model.encode([descriptions[k] for k in ids_list], batch_size=32,
                                show_progress_bar=True, normalize_embeddings=True).astype(np.float32)
    np.savez(EMB_CACHE, ids=np.array(ids_list), vecs=vecs)
    id2emb = {k: vecs[i] for i, k in enumerate(ids_list)}
    del st_model; gc.collect()

TEXT_DIM = next(iter(id2emb.values())).shape[0]
print(f'{len(id2emb)} embeddings x {TEXT_DIM}-dim')

3645 embeddings x 768-dim


In [ ]:
class MultimodalFoodDataset(Dataset):
    def __init__(self, df, transforms, id2emb, is_train=True):
        self.df         = df.reset_index(drop=True)
        self.transforms = transforms
        self.id2emb     = id2emb
        self.is_train   = is_train
        self.fallback   = np.zeros(TEXT_DIM, dtype=np.float32)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row.path, cv2.IMREAD_COLOR)
        if img is None: raise FileNotFoundError(row.path)
        img      = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img      = self.transforms(image=img)['image']
        text_emb = torch.tensor(self.id2emb.get(str(row.image_id), self.fallback), dtype=torch.float32)
        if self.is_train:
            return img, text_emb, torch.tensor(float(row.target_log), dtype=torch.float32)
        return img, text_emb, str(row.image_id)

def make_loader(df, transforms, batch_size, is_train, shuffle, drop_last=False):
    ds = MultimodalFoodDataset(df, transforms, id2emb, is_train=is_train)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=drop_last)

In [ ]:
BACKBONES_WITH_IMG_SIZE = ('swin', 'vit', 'deit', 'beit')

class MultimodalRegressor(nn.Module):
    def __init__(self, backbone_name, img_size, text_dim, dropout=0.25):
        super().__init__()
        needs_img_size = any(k in backbone_name for k in BACKBONES_WITH_IMG_SIZE)
        extra = {'img_size': img_size} if needs_img_size else {}
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, num_classes=0, global_pool='avg', **extra)
        fused_dim = self.backbone.num_features + text_dim
        self.head = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 128),       nn.GELU(), nn.Dropout(dropout / 2),
            nn.Linear(128, 1),
        )
        print(f'  fused={fused_dim}')

    def forward(self, img, text_emb):
        return self.head(torch.cat([self.backbone(img), text_emb], dim=1)).squeeze(1)

def get_optimizer_llrd(model, base_lr, wd, n_groups=4, decay=0.25):
    bp = [(n,p) for n,p in model.backbone.named_parameters() if p.requires_grad]
    hp = [(n,p) for n,p in model.head.named_parameters()    if p.requires_grad]
    gs = max(1, len(bp)//n_groups); pg = []
    for g in range(n_groups):
        s = g*gs; e = (g+1)*gs if g < n_groups-1 else len(bp)
        lr_g = base_lr * (decay**(n_groups-1-g)); chunk = bp[s:e]
        wd_  = [p for n,p in chunk if p.ndim>1 and not n.endswith('bias') and 'norm' not in n.lower()]
        nwd  = [p for n,p in chunk if p.ndim<=1 or n.endswith('bias') or 'norm' in n.lower()]
        if wd_:  pg.append({'params':wd_,  'lr':lr_g, 'weight_decay':wd})
        if nwd:  pg.append({'params':nwd,  'lr':lr_g, 'weight_decay':0.})
    hwd  = [p for n,p in hp if p.ndim>1 and not n.endswith('bias') and 'norm' not in n.lower()]
    hnwd = [p for n,p in hp if p.ndim<=1 or n.endswith('bias') or 'norm' in n.lower()]
    if hwd:  pg.append({'params':hwd,  'lr':base_lr, 'weight_decay':wd})
    if hnwd: pg.append({'params':hnwd, 'lr':base_lr, 'weight_decay':0.})
    return torch.optim.AdamW(pg)

In [ ]:
def preds_to_calories(p): return np.clip(np.expm1(p), PRED_MIN, PRED_MAX)

MIXUP_ALPHA = 0.4; AUG_MIX_PROB = 0.3

def mixup(imgs, texts, targets):
    lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    return lam*imgs+(1-lam)*imgs[idx], lam*texts+(1-lam)*texts[idx], lam*targets+(1-lam)*targets[idx]

def train_one_epoch(model, loader, optimizer, scheduler, scaler, criterion):
    model.train(); total = 0.; optimizer.zero_grad(set_to_none=True)
    for imgs, texts, targets in loader:
        imgs    = imgs.to(device, non_blocking=True)
        texts   = texts.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        if random.random() < AUG_MIX_PROB:
            imgs, texts, targets = mixup(imgs, texts, targets)
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            loss = criterion(model(imgs, texts), targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
        if scheduler: scheduler.step()
        total += float(loss.item())
    return total / len(loader)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval(); losses, preds = [], []
    for imgs, texts, targets in loader:
        imgs    = imgs.to(device, non_blocking=True)
        texts   = texts.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            out = model(imgs, texts); loss = criterion(out, targets)
        losses.append(float(loss.item())); preds.append(out.float().cpu().numpy())
    return float(np.mean(losses)), np.concatenate(preds)

@torch.no_grad()
def predict(model, loader):
    model.eval(); all_ids, all_preds = [], []
    for imgs, texts, ids in loader:
        imgs  = imgs.to(device, non_blocking=True)
        texts = texts.to(device, non_blocking=True)
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            out      = model(imgs, texts)
            out_flip = model(torch.flip(imgs, dims=[3]), texts)
        all_preds.append(((out+out_flip)/2).float().cpu().numpy())
        all_ids.extend(ids if isinstance(ids, list) else list(ids))
    return all_ids, np.concatenate(all_preds)

In [ ]:
MODELS = [
    ('convnextv2_tiny.fcmae_ft_in22k_in1k',        224, 'convnextv2_tiny_mm',  128),
    ('tf_efficientnetv2_s.in21k_ft_in1k',           224, 'efficientnetv2_s_mm', 128),
    ('deit3_small_patch16_224.fb_in22k_ft_in1k',    224, 'deit3_small_mm',      128),
]

LR=2e-4; WD=1e-2; DROPOUT=0.25
EPOCHS    = DEBUG_EPOCHS if DEBUG else 100
MA_WINDOW = 1  if DEBUG else 10
PATIENCE  = 1  if DEBUG else 15

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f'VRAM libre: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB / {torch.cuda.mem_get_info()[1]/1e9:.1f} GB')

VRAM libre: 42.0 GB / 42.4 GB


In [ ]:
results = {}

for backbone_name, img_size, exp_name, bs in MODELS:
    print(f'\n{exp_name}')
    ckpt_path  = CKPT_DIR / f'{exp_name}_best.pth'
    preds_path = OUT_DIR  / f'{exp_name}_preds.npz'

    if not DEBUG and ckpt_path.exists() and preds_path.exists():
        saved = np.load(preds_path)
        results[exp_name] = {
            'val_mae':   mean_absolute_error(df_val['calories'].values,        preds_to_calories(saved['val_preds'])),
            'test_mae':  mean_absolute_error(df_test_final['calories'].values, preds_to_calories(saved['test_final_preds'])),
            'best_epoch': int(torch.load(ckpt_path, map_location='cpu', weights_only=False)['epoch']),
        }
        print(f'  skip | val={results[exp_name]["val_mae"]:.2f} test={results[exp_name]["test_mae"]:.2f} epoch={results[exp_name]["best_epoch"]}')
        continue

    train_tfm = A.Compose([
        A.RandomResizedCrop(size=(img_size,img_size), scale=(0.65,1.0), ratio=(0.80,1.20), interpolation=cv2.INTER_AREA),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.08, p=0.6),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])
    val_tfm = A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
        A.CenterCrop(height=img_size, width=img_size),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])

    train_loader = make_loader(df_train, train_tfm, bs, is_train=True,  shuffle=True,  drop_last=True)
    val_loader   = make_loader(df_val,   val_tfm,   bs, is_train=True,  shuffle=False)

    model     = MultimodalRegressor(backbone_name, img_size, TEXT_DIM, DROPOUT).to(device)
    optimizer = get_optimizer_llrd(model, base_lr=LR, wd=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS*len(train_loader), eta_min=1e-7)
    scaler    = GradScaler(enabled=torch.cuda.is_available())
    criterion = nn.HuberLoss(delta=0.5)

    best_ma = float('inf'); best_mae = float('inf'); wait = 0
    mae_history = deque(maxlen=MA_WINDOW)

    for epoch in range(1, EPOCHS+1):
        t0         = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, criterion)
        _, val_log = evaluate(model, val_loader, criterion)
        val_mae    = mean_absolute_error(df_val['calories'].values, preds_to_calories(val_log))
        mae_history.append(val_mae)
        ma = float(np.mean(mae_history))

        if val_mae < best_mae:
            best_mae = val_mae
            torch.save({'model_state': model.state_dict(), 'epoch': epoch,
                        'best_mae': best_mae, 'backbone': backbone_name, 'img_size': img_size}, ckpt_path)

        print(f'  {epoch:03d}/{EPOCHS} | train={train_loss:.4f} val={val_mae:.2f} ma={ma:.2f} best={best_mae:.2f} | {time.time()-t0:.0f}s', flush=True)

        if len(mae_history) == MA_WINDOW:
            if ma < best_ma: best_ma = ma; wait = 0
            else:
                wait += 1
                if wait >= PATIENCE:
                    print(f'  early stop ep={epoch} best={best_mae:.2f}'); break

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])

    test_loader   = make_loader(df_test_final, val_tfm, bs, is_train=True,  shuffle=False)
    kaggle_loader = make_loader(test_df,       val_tfm, bs, is_train=False, shuffle=False)

    _, vpl = evaluate(model, val_loader,  criterion)
    _, tpl = evaluate(model, test_loader, criterion)
    kids, kpl = predict(model, kaggle_loader)
    np.savez(preds_path, val_preds=vpl, test_final_preds=tpl,
             kaggle_ids=np.array(kids), kaggle_preds=preds_to_calories(kpl))

    results[exp_name] = {
        'val_mae':   mean_absolute_error(df_val['calories'].values,        preds_to_calories(vpl)),
        'test_mae':  mean_absolute_error(df_test_final['calories'].values, preds_to_calories(tpl)),
        'best_epoch': ckpt['epoch'],
    }
    print(f'  val={results[exp_name]["val_mae"]:.2f} test={results[exp_name]["test_mae"]:.2f} epoch={ckpt["epoch"]}')

    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


convnextv2_tiny_mm


model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

  fused=1536
  001/100 | train=1.1078 val=256.60 ma=256.60 best=256.60 | 91s
  002/100 | train=0.2401 val=218.81 ma=237.71 best=218.81 | 53s
  003/100 | train=0.1518 val=175.41 ma=216.94 best=175.41 | 52s
  004/100 | train=0.1389 val=210.03 ma=215.21 best=175.41 | 52s
  005/100 | train=0.1321 val=135.66 ma=199.30 best=135.66 | 53s
  006/100 | train=0.1022 val=163.04 ma=193.26 best=135.66 | 52s
  007/100 | train=0.1227 val=132.38 ma=184.56 best=132.38 | 51s
  008/100 | train=0.0993 val=132.98 ma=178.11 best=132.38 | 53s
  009/100 | train=0.1159 val=105.55 ma=170.05 best=105.55 | 53s
  010/100 | train=0.0940 val=114.82 ma=164.53 best=105.55 | 51s
  011/100 | train=0.0849 val=122.43 ma=151.11 best=105.55 | 52s
  012/100 | train=0.0903 val=154.16 ma=144.65 best=105.55 | 52s
  013/100 | train=0.0886 val=121.92 ma=139.30 best=105.55 | 51s
  014/100 | train=0.0773 val=161.81 ma=134.47 best=105.55 | 52s
  015/100 | train=0.0755 val=113.87 ma=132.30 best=105.55 | 51s
  016/100 | train=0.0816 va

model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

  fused=2048
  001/100 | train=1.3222 val=1081.84 ma=1081.84 best=1081.84 | 107s
  002/100 | train=0.3066 val=187.58 ma=634.71 best=187.58 | 51s
  003/100 | train=0.1772 val=253.45 ma=507.62 best=187.58 | 52s
  004/100 | train=0.1522 val=166.78 ma=422.41 best=166.78 | 52s
  005/100 | train=0.1399 val=208.89 ma=379.71 best=166.78 | 51s
  006/100 | train=0.1519 val=192.14 ma=348.45 best=166.78 | 51s
  007/100 | train=0.1368 val=160.85 ma=321.65 best=160.85 | 51s
  008/100 | train=0.1495 val=178.83 ma=303.80 best=160.85 | 52s
  009/100 | train=0.1333 val=196.43 ma=291.87 best=160.85 | 51s
  010/100 | train=0.1092 val=166.56 ma=279.34 best=160.85 | 52s
  011/100 | train=0.1118 val=178.21 ma=188.97 best=160.85 | 52s
  012/100 | train=0.1071 val=174.05 ma=187.62 best=160.85 | 52s
  013/100 | train=0.0952 val=171.10 ma=179.38 best=160.85 | 51s
  014/100 | train=0.0978 val=173.38 ma=180.04 best=160.85 | 51s
  015/100 | train=0.0969 val=163.63 ma=175.52 best=160.85 | 52s
  016/100 | train=0.094

model.safetensors:   0%|          | 0.00/88.3M [00:00<?, ?B/s]

  fused=1152
  001/100 | train=1.1549 val=304.64 ma=304.64 best=304.64 | 52s
  002/100 | train=0.2664 val=219.90 ma=262.27 best=219.90 | 51s
  003/100 | train=0.1632 val=202.61 ma=242.39 best=202.61 | 51s
  004/100 | train=0.1340 val=174.20 ma=225.34 best=174.20 | 52s
  005/100 | train=0.1400 val=164.30 ma=213.13 best=164.30 | 51s
  006/100 | train=0.1499 val=163.70 ma=204.89 best=163.70 | 51s
  007/100 | train=0.1084 val=183.04 ma=201.77 best=163.70 | 51s
  008/100 | train=0.1021 val=157.48 ma=196.24 best=157.48 | 53s
  009/100 | train=0.1223 val=141.10 ma=190.11 best=141.10 | 52s
  010/100 | train=0.1098 val=135.29 ma=184.63 best=135.29 | 51s
  011/100 | train=0.0896 val=131.64 ma=167.33 best=131.64 | 51s
  012/100 | train=0.0944 val=123.94 ma=157.73 best=123.94 | 52s
  013/100 | train=0.0781 val=135.37 ma=151.01 best=123.94 | 51s
  014/100 | train=0.0741 val=117.77 ma=145.36 best=117.77 | 53s
  015/100 | train=0.0709 val=110.26 ma=139.96 best=110.26 | 51s
  016/100 | train=0.0707 va

In [ ]:
from scipy.optimize import minimize
from sklearn.ensemble import RandomForestRegressor

npz_files   = sorted(OUT_DIR.glob('*_preds.npz'))
model_names = []; val_preds_l = []; test_preds_l = []; kag_preds_l = []; kaggle_ids = None

for f in npz_files:
    data = np.load(f, allow_pickle=True)
    vp = preds_to_calories(data['val_preds'])
    tp = preds_to_calories(data['test_final_preds'])
    kp = data['kaggle_preds']
    val_preds_l.append(vp); test_preds_l.append(tp); kag_preds_l.append(kp)
    if kaggle_ids is None: kaggle_ids = list(data['kaggle_ids'])
    model_names.append(f.stem.replace('_preds',''))
    print(f'  {f.stem:<35} val={mean_absolute_error(df_val["calories"].values, vp):.2f} test={mean_absolute_error(df_test_final["calories"].values, tp):.2f}')

val_preds  = np.stack(val_preds_l)
test_preds = np.stack(test_preds_l)
kag_preds  = np.stack(kag_preds_l)
y_val  = df_val['calories'].values
y_test = df_test_final['calories'].values
M      = len(model_names)

  convnextv2_tiny_mm_preds            val=91.20 test=79.51
  deit3_small_mm_preds                val=96.62 test=84.50
  efficientnetv2_s_mm_preds           val=125.09 test=131.35


In [ ]:
def blend_mae(w):
    w = np.abs(w); w = w / w.sum()
    return mean_absolute_error(y_val, (val_preds * w[:, None]).sum(axis=0))

res  = minimize(blend_mae, np.ones(M)/M, method='Nelder-Mead',
                options={'maxiter': 10000, 'xatol': 1e-6, 'fatol': 1e-6})
w_nm = np.abs(res.x); w_nm /= w_nm.sum()

blend_val_nm  = (val_preds  * w_nm[:, None]).sum(axis=0)
blend_test_nm = (test_preds * w_nm[:, None]).sum(axis=0)
blend_kag_nm  = (kag_preds  * w_nm[:, None]).sum(axis=0)
mae_val_nm    = mean_absolute_error(y_val,  blend_val_nm)
mae_test_nm   = mean_absolute_error(y_test, blend_test_nm)

for name, w in zip(model_names, w_nm):
    print(f'  {name:<35} {w:.4f}')
print(f'NM  val={mae_val_nm:.2f} test={mae_test_nm:.2f}')

  convnextv2_tiny_mm                  0.6366
  deit3_small_mm                      0.3634
  efficientnetv2_s_mm                 0.0000
NM  val=86.97 test=75.63


In [ ]:
rf = RandomForestRegressor(n_estimators=500, max_depth=6, random_state=SEED, n_jobs=-1)
rf.fit(val_preds.T, y_val)

blend_val_rf  = rf.predict(val_preds.T)
blend_test_rf = rf.predict(test_preds.T)
blend_kag_rf  = rf.predict(kag_preds.T)
mae_val_rf    = mean_absolute_error(y_val,  blend_val_rf)
mae_test_rf   = mean_absolute_error(y_test, blend_test_rf)

print(f'RF  val={mae_val_rf:.2f} test={mae_test_rf:.2f}')
for name, imp in sorted(zip(model_names, rf.feature_importances_), key=lambda x: -x[1]):
    print(f'  {name:<35} {imp:.4f}')

RF  val=52.76 test=80.72
  convnextv2_tiny_mm                  0.6590
  deit3_small_mm                      0.2244
  efficientnetv2_s_mm                 0.1166


In [ ]:
print(f'{"model":<35} {"val":>8} {"test":>8}')
for name, r in results.items():
    print(f'{name:<35} {r["val_mae"]:>8.2f} {r["test_mae"]:>8.2f}  (ep {r["best_epoch"]})')
print(f'{"NM blend":<35} {mae_val_nm:>8.2f} {mae_test_nm:>8.2f}')
print(f'{"RF blend":<35} {mae_val_rf:>8.2f} {mae_test_rf:>8.2f}')

if mae_test_nm <= mae_test_rf:
    best_kag = blend_kag_nm; best_tag = f'nm_{mae_test_nm:.2f}'
else:
    best_kag = blend_kag_rf; best_tag = f'rf_{mae_test_rf:.2f}'

sub = pd.DataFrame({'image_id': kaggle_ids, 'calories': np.clip(best_kag, PRED_MIN, PRED_MAX).round(2)})
sub.to_csv(WORK / f'submission_{best_tag}.csv', index=False)
print(f'\nsubmission_{best_tag}.csv')
print(sub.head())

if not DEBUG:
    best_epochs = {name: r['best_epoch'] for name, r in results.items()}
    nm_weights  = {name: float(w) for name, w in zip(model_names, w_nm)}
    (WORK / 'best_epochs.json').write_text(json.dumps(best_epochs, indent=2))
    (WORK / 'nelder_mead_weights.json').write_text(json.dumps(nm_weights, indent=2))
    print('best_epochs.json:', best_epochs)

model                                    val     test
convnextv2_tiny_mm                     91.20    79.51  (ep 72)
efficientnetv2_s_mm                   125.09   131.35  (ep 60)
deit3_small_mm                         96.62    84.50  (ep 36)
NM blend                               86.97    75.63
RF blend                               52.76    80.72

submission_nm_75.63.csv
    image_id  calories
0  test_0000     91.30
1  test_0001    188.48
2  test_0002    166.04
3  test_0003    186.54
4  test_0004    205.76
best_epochs.json: {'convnextv2_tiny_mm': 72, 'efficientnetv2_s_mm': 60, 'deit3_small_mm': 36}


In [ ]:
from scipy.optimize import minimize
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
import numpy as np
from pathlib import Path
from google.colab import files

OUT_BASE = Path('/content/drive/MyDrive/food_step1/outputs')
OUT_TINY = Path('/content/drive/MyDrive/food_step1_tiny/outputs')

PRED_MIN=30.0; PRED_MAX=3600.0
def ptc(p): return np.clip(np.expm1(p), PRED_MIN, PRED_MAX)

all_names = []; val_l = []; test_l = []; kag_l = []; kaggle_ids = None

for out_dir in [OUT_BASE, OUT_TINY]:
    for f in sorted(out_dir.glob('*_preds.npz')):
        data = np.load(f, allow_pickle=True)
        vp = ptc(data['val_preds'])
        tp = ptc(data['test_final_preds'])
        kp = data['kaggle_preds']
        val_l.append(vp); test_l.append(tp); kag_l.append(kp)
        if kaggle_ids is None: kaggle_ids = list(data['kaggle_ids'])
        name = f'{out_dir.parent.name}/{f.stem.replace("_preds","")}'
        all_names.append(name)
        print(f'  {name:<50} val={mean_absolute_error(df_val["calories"].values, vp):.2f} test={mean_absolute_error(df_test_final["calories"].values, tp):.2f}')

val_all  = np.stack(val_l)
test_all = np.stack(test_l)
kag_all  = np.stack(kag_l)
y_val    = df_val['calories'].values
y_test   = df_test_final['calories'].values
M        = len(all_names)

# NM
def blend_mae(w):
    w = np.abs(w); w /= w.sum()
    return mean_absolute_error(y_val, (val_all * w[:,None]).sum(axis=0))
res  = minimize(blend_mae, np.ones(M)/M, method='Nelder-Mead', options={'maxiter':20000,'xatol':1e-6,'fatol':1e-6})
w_nm = np.abs(res.x); w_nm /= w_nm.sum()
blend_kag_nm  = (kag_all * w_nm[:,None]).sum(axis=0)
mae_val_nm    = mean_absolute_error(y_val,  (val_all  * w_nm[:,None]).sum(axis=0))
mae_test_nm   = mean_absolute_error(y_test, (test_all * w_nm[:,None]).sum(axis=0))

# RF
rf = RandomForestRegressor(n_estimators=500, max_depth=2, random_state=42, n_jobs=-1)
rf.fit(val_all.T, y_val)
mae_val_rf   = mean_absolute_error(y_val,  rf.predict(val_all.T))
mae_test_rf  = mean_absolute_error(y_test, rf.predict(test_all.T))
blend_kag_rf = rf.predict(kag_all.T)

# Decision Tree
dt = DecisionTreeRegressor(max_depth=2, random_state=42)
dt.fit(val_all.T, y_val)
mae_val_dt   = mean_absolute_error(y_val,  dt.predict(val_all.T))
mae_test_dt  = mean_absolute_error(y_test, dt.predict(test_all.T))
blend_kag_dt = dt.predict(kag_all.T)

# Polynomial degree=2 + Ridge
poly = PolynomialFeatures(degree=2, include_bias=False)
X_val_p  = poly.fit_transform(val_all.T)
X_test_p = poly.transform(test_all.T)
X_kag_p  = poly.transform(kag_all.T)
pr = Ridge(alpha=1.0)
pr.fit(X_val_p, y_val)
mae_val_poly   = mean_absolute_error(y_val,  pr.predict(X_val_p))
mae_test_poly  = mean_absolute_error(y_test, pr.predict(X_test_p))
blend_kag_poly = pr.predict(X_kag_p)

print(f'\n{"method":<20} {"val":>8} {"test":>8}')
print(f'{"NM":<20} {mae_val_nm:>8.2f} {mae_test_nm:>8.2f}')
print(f'{"RF (depth=2)":<20} {mae_val_rf:>8.2f} {mae_test_rf:>8.2f}')
print(f'{"DT (depth=2)":<20} {mae_val_dt:>8.2f} {mae_test_dt:>8.2f}')
print(f'{"Poly+Ridge":<20} {mae_val_poly:>8.2f} {mae_test_poly:>8.2f}')

# meilleur sur test
results_blend = {
    'nm':   (mae_test_nm,   blend_kag_nm),
    'rf':   (mae_test_rf,   blend_kag_rf),
    'dt':   (mae_test_dt,   blend_kag_dt),
    'poly': (mae_test_poly, blend_kag_poly),
}
best_name = min(results_blend, key=lambda k: results_blend[k][0])
best_preds = results_blend[best_name][1]
print(f'\nbest: {best_name} (test={results_blend[best_name][0]:.2f})')

sub_path = '/content/drive/MyDrive/food_step1/submission_best_blend.csv'
sub = pd.DataFrame({'image_id': kaggle_ids, 'calories': np.clip(best_preds, PRED_MIN, PRED_MAX).round(2)})
sub.to_csv(sub_path, index=False)
print(f'sauvegardé: {sub_path}')
files.download(sub_path)

  food_step1/convnextv2_base_mm                      val=72.89 test=54.14
  food_step1/deit3_base_mm                           val=86.41 test=84.63
  food_step1/efficientnetv2_m_mm                     val=116.62 test=109.67
  food_step1_tiny/convnextv2_tiny_mm                 val=91.20 test=79.51
  food_step1_tiny/deit3_small_mm                     val=96.62 test=84.50
  food_step1_tiny/efficientnetv2_s_mm                val=125.09 test=131.35

method                    val     test
NM                      69.25    58.25
RF (depth=2)           110.52   113.91
DT (depth=2)           122.78   133.73
Poly+Ridge              66.26    90.87

best: nm (test=58.25)
sauvegardé: /content/drive/MyDrive/food_step1/submission_best_blend.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>